# 1.b - AquaCrop rainfed maize on every land cell of a region, against the SVM suitability

The chain of the pilot 1.a on every land cell of `REGION` (Europe by default), cells where maize is grown today and
cells where it is not, to compare the rainfed yield of AquaCrop with the SVM probability over the whole map. Model:
FAO **AquaCrop 7.3** stand-alone (KU Leuven build, release v7.3_typofix, downloaded by `aquacrop_utils.ensure_installed`),
crop file `MaizeGDD.CRO` of the FAO database. Writes only in `outputs/aquacrop/maize_<region>/` and in a work folder
(`WORK`).

**Climate: the ERA5 of the SVM.** The daily ERA5 on the SVM (EC-Earth3) grid, dataset `era5`, read cell by cell as
11.a: both models see the same climate, so the differences come from the models; it is also the analogue of the
EC-Earth3 grid boxes of the runs (land and sea mixed in coastal cells, a limitation common to both models and to the
GCM).

**Inputs per cell** (SVM grid, EC-Earth3 0.7 deg):
- climate: daily ERA5 tasmax, tasmin, pr of the cell (SVM grid); ET0 of Hargreaves (as 1.1.d / 11.a, no
  radiation); CO2 of the FAO Mauna Loa file (observed concentration of each year);
- sowing: GGCMI phase 3 planting day of the cell (nearest grid point with a value where the cell has none);
- cultivar: `MaizeGDD.CRO` is a maize of Davis (California) needing 1700 degree-days to maturity. As gridded crop
  models do (GGCMI), the cultivar is adapted to the local season: the degree-days from the GGCMI planting to the GGCMI
  maturity day (mean over the seasons, AquaCrop method 3) replace the 1700, every degree-day duration of the crop file
  but emergence scaled by the same factor and the canopy coefficients per degree-day divided by it
  (`aquacrop_utils.write_crop_variant`); the physiology (water productivity, harvest index, stress responses) is that
  of the FAO file. A cultivar is a property of the crop, not of the climate: where maize is not grown, the nearest GGCMI
  season applied to a colder climate would give cultivars that do not exist (140-230 degree-days in Lapland), so the
  cultivar is not earlier than the earliest one grown today (5th percentile of the local degree-days of the cells where
  maize is grown);
- soil: the layers of HWSD v2 (D1-D7, down to 2 m or to the first layer without data in shallow soils) of the dominant
  component of the dominant soil mapping unit of the cell, grouped into 5 horizons (0-20, 20-40, 40-80, 80-150,
  150-200 cm); saturation, field capacity, wilting point and saturated conductivity of the fine earth from sand,
  clay, organic matter and bulk density with the pedotransfer functions of Saxton & Rawls (2006), as regional AquaCrop
  set-ups use HWSD with pedotransfer functions (de Roos et al. 2021, GMD 14, 7309); gravel from the HWSD coarse
  fragments (AquaCrop reduces the water storage itself); curve number, readily evaporable water and capillary-rise
  parameters with the rules of the FAO database (checked on its 12 default soils); where HWSD has the bedrock below
  the last layer (shallow soils), a horizon of penetrability 0 % down to 2 m, the restrictive layer of AquaCrop (the
  thickness of the horizons alone does not limit the roots: tested);
- no irrigation, no field management (no fertility stress), no groundwater.

**Seasons.** One continuous water balance per cell (`KeepSWC`: each run starts with the soil water left by the
previous one); run k from 30 days before sowing to 31 days before the next sowing; the first season starts at field
capacity and is dropped (spin-up). The crop cycle ends at maturity (growing degree-days of the crop file) as computed
by AquaCrop.

**AquaCrop as it is.** The results of AquaCrop are kept as they are, including where the crop cannot reach maturity
(AquaCrop then prolongs the cycle to the end of the run and still reports a yield; tested: 9.3 t/ha with 1450
degree-days a year for 1700 required): the comparison is between the two models as they work. With the cultivar of
the local season this concerns mainly the cells where maize is not grown. The degree-days of
each season (AquaCrop method 3, from sowing to the end of the run) are reported for information only. Where no
degree-day is accumulated at all (ice), the programme does not end: every cell runs alone with a time limit, and a cell
that does not finish has no result (`no result`).

**Cells.** Every land cell of `REGION` with an SVM prediction on ERA5; grown = presence of the training route.

**Outputs.** Per cell: mean rainfed dry yield over the seasons, its variability, failed seasons, cycle length, water
and temperature stress (table and NetCDF on the SVM grid); maps of the SVM probability and of the yield; rank
correlation between them; how well each separates today's maize cells from the others (AUC).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
import sys, glob, sqlite3, tempfile, time
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
sys.path.insert(0, str(BASE)); sys.path.insert(0, str(BASE / "2_AQUACROP"))
import crop_utils as cu
import aquacrop_utils as aq
CFG = cu.load_config()

CROP = "Maize_rf"
CROP_FILE = "MaizeGDD.CRO"                       # FAO database, crop cycle in growing degree-days
GGCMI = "mai"                                    # GGCMI phase 3 calendar
CELL_AREA_THR = 0.0005                           # presence route of the SVM (cellareafrac 0.05 %)
SVM_SET = "A_60d_no_rad"                         # 4.g predictions (scenario of the training climate, CROP_REF)
YEARS = (1981, 2014)                             # harvest years; the first season is the spin-up
REGION = tuple(float(x) for x in os.environ.get("AQUACROP_REGION", "34,72,-25,45").split(","))   # lat0, lat1, lon0, lon1
REGION_NAME = os.environ.get("AQUACROP_REGION_NAME", "europe")
SOW_LEAD = 30                                    # days of simulation before sowing
WORK = Path(os.environ.get("AQUACROP_WORK", Path(tempfile.gettempdir()) / f"aquacrop_maize_{REGION_NAME}"))
N_WORKERS = int(os.environ.get("AQUACROP_WORKERS", str(max(1, (os.cpu_count() or 2) - 1))))   # cells run in parallel
TIMEOUT = 120                                   # s per cell (only to stop a cell where AquaCrop does not end)

ROUTE = cu.route(CELL_AREA_THR)
OUT = cu.out_dir(CFG, "aquacrop", f"maize_{REGION_NAME}")
GRID = cu.svm_grid(CFG)
print(f"AquaCrop {aq.AQ_VERSION}: {aq.EXE} | crop {CROP_FILE} | region {REGION_NAME} {REGION} | {N_WORKERS} workers | "
      f"work {WORK} | written in {OUT}")

## 1. Cells: every land cell of the region with an SVM prediction; daily ERA5 of the cells

In [ ]:
with xr.open_dataset(cu.out_dir(CFG, "4_svm_predict", ROUTE, SVM_SET) / f"suitability_{CROP}.nc") as ds:
    P = ds["suit_proba"].sel(scenario=cu.REF_LABEL).values
cm = cu.crop_mask(CFG, CROP)
PRES = (cm["presence"].values == 1) & (cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR)
LAND = cu.land_mask(CFG).values
la, lo = GRID["lat"], GRID["lon"]
inreg = ((la >= REGION[0]) & (la <= REGION[1]))[:, None] & ((lo >= REGION[2]) & (lo <= REGION[3]))[None, :]
CI, CJ = np.nonzero(LAND & inreg & np.isfinite(P))
CELLS = pd.DataFrame({"i": CI, "j": CJ, "lat": la[CI], "lon": lo[CJ], "svm_p": P[CI, CJ],
                      "group": np.where(PRES[CI, CJ], "grown", "not grown")},
                     index=pd.Index([f"c{i:03d}_{j:03d}" for i, j in zip(CI, CJ)], name="cell"))
print(f"{len(CELLS)} land cells in {REGION_NAME} ({(CELLS['group'] == 'grown').sum()} grown)")

t0 = time.time()
CL = {}
for v in ("tasmax", "tasmin", "pr"):                       # as read_daily of 11.a
    da = cu.read_climate(CFG, cu.CROP_REF, v, GRID)   # ERA5 or W5E5 (CROP_REF), as the SVM
    CL[v] = cu.land_series(da, (CI, CJ))
T = pd.DatetimeIndex(da["time"].values).normalize()
print(f"{cu.REF_LABEL} {T[0].date()} -> {T[-1].date()}, {len(CELLS)} cells ({(time.time() - t0) / 60:.1f} min)")
COL = {cell: k for k, cell in enumerate(CELLS.index)}

## 2. Inputs per cell: daily climate, sowing day, soil

In [ ]:
def cell_box(i, j):
    return (GRID["lat_edges"][i], GRID["lat_edges"][i + 1], GRID["lon_edges"][j], GRID["lon_edges"][j + 1])


def cell_climate(cell):
    k = COL[cell]
    return T, {v: CL[v][:, k].astype(float) for v in CL}


# sowing day: GGCMI phase 3 planting day (nearest point with a value)
with xr.open_dataset(glob.glob(str(Path(cu.data_files(CFG, "ggcmi_calendar")[0]).parent / f"{GGCMI}_rf_ggcmi_crop_calendar_phase3*.nc4"))[0],
                     decode_times=False) as ds:
    G = ds["planting_day"].values; GM = ds["maturity_day"].values; glat, glon = ds["lat"].values, ds["lon"].values
gi, gj = np.nonzero(np.isfinite(G) & np.isfinite(GM))


def sowing_doy(lat, lon):
    # GGCMI planting and maturity day of the nearest point with both, distance (deg)
    k = np.argmin((glat[gi] - lat) ** 2 + ((glon[gj] - lon + 180) % 360 - 180) ** 2 * np.cos(np.radians(lat)) ** 2)
    return (int(round(G[gi[k], gj[k]])), int(round(GM[gi[k], gj[k]])),
            float(np.hypot(glat[gi[k]] - lat, glon[gj[k]] - lon)))


# soil: HWSD v2 layers of the dominant component of the dominant soil mapping unit, Saxton & Rawls horizons
import rasterio
from rasterio.windows import from_bounds
HW_RASTER = cu.data_file(CFG, "hwsd_raster")
HW_DB = Path(cu.data_file(CFG, "hwsd_mdb")).with_suffix(".sqlite")
con = cu.hwsd_connect(HW_DB if HW_DB.exists() else cu.data_file(CFG, "hwsd_mdb"))
LAY = pd.read_sql("SELECT HWSD2_SMU_ID, SEQUENCE, SHARE, LAYER, TOPDEP, BOTDEP, SAND, SILT, CLAY, ORG_CARBON, BULK, COARSE "
                  "FROM HWSD2_LAYERS", con)
for k in ("SAND", "SILT", "CLAY", "ORG_CARBON", "BULK", "COARSE", "TOPDEP", "BOTDEP"):
    LAY[k] = pd.to_numeric(LAY[k], errors="coerce")
# negative values kept: -7 = below the bedrock (read by aquacrop_utils.hwsd_profile), -3 / -9 = no data
D1OK = LAY[(LAY["LAYER"] == "D1") & (LAY["SAND"] >= 0)]              # components with a topsoil description
DOMC = (D1OK.sort_values(["HWSD2_SMU_ID", "SHARE", "SEQUENCE"], ascending=[True, False, True])
        .drop_duplicates("HWSD2_SMU_ID")[["HWSD2_SMU_ID", "SEQUENCE"]])
LAYG = LAY.groupby(["HWSD2_SMU_ID", "SEQUENCE"])
DOM_LAYERS = {smu: LAYG.get_group((smu, seq)) for smu, seq in DOMC.itertuples(index=False)}


def soil_profile(i, j):
    # dominant soil mapping unit with layer data in the cell -> AquaCrop horizons (None: water, ice, rock)
    la0, la1, lo0, lo1 = cell_box(i, j)
    with rasterio.open(HW_RASTER) as r:
        a = r.read(1, window=from_bounds(lo0, la0, lo1, la1, r.transform))
    for smu in pd.Series(a[(a != 65535) & (a > 0)]).value_counts().index:
        if int(smu) in DOM_LAYERS:
            hz = aq.hwsd_profile(DOM_LAYERS[int(smu)])
            if hz:
                return int(smu), hz
    return None, None


GP = aq.crop_gdd_parameters(CROP_FILE)
PREP, INFO = {}, []
t0 = time.time()
for cell, c in CELLS.iterrows():
    t, X = cell_climate(cell)
    keep = (t.year >= YEARS[0] - 1) & (t.year <= YEARS[1] + 1)
    t = t[keep]; X = {v: x[keep] for v, x in X.items()}
    eto = aq.eto_hargreaves(X["tasmax"], X["tasmin"], c.lat, t.dayofyear.values)
    sow, mat, dist = sowing_doy(c.lat, c.lon)
    smu, hz = soil_profile(c.i, c.j)
    # one run per season: from SOW_LEAD days before sowing to the day before the next run (continuous water balance)
    sowd = [aq.daynr(date(y, 1, 1)) + sow - 1 for y in range(YEARS[0], YEARS[1] + 1)]
    runs = [{"sim1": s - SOW_LEAD, "simN": (sowd[k + 1] - SOW_LEAD - 1) if k + 1 < len(sowd) else s + 365 - SOW_LEAD - 1,
             "crop1": s, "cropN": s + 1, "keep": True} for k, s in enumerate(sowd)]
    d0, dN = aq.daynr(t[0].date()), aq.daynr(t[-1].date())
    runs = [r for r in runs if r["sim1"] >= d0 and r["simN"] <= dN]
    gdd = aq.daily_gdd(X["tasmin"], X["tasmax"], GP["tbase"], GP["tupper"])
    # cultivar of the local season: degree-days from the GGCMI planting to the GGCMI maturity day, mean of the seasons
    glen = (mat - sow) % 365 or 365
    gdd_local = float(np.mean([gdd[r["crop1"] - d0:r["crop1"] - d0 + glen].sum() for r in runs]))
    for r in runs:                                   # degree-days from sowing to the end of the run, and of the run (information)
        r["gdd_season"] = float(gdd[r["crop1"] - d0:r["simN"] - d0 + 1].sum())
        r["gdd_run"] = float(gdd[r["sim1"] - d0:r["simN"] - d0 + 1].sum())
    PREP[cell] = {"first": t[0].date(), "X": X, "eto": eto, "runs": runs, "hz": hz, "title": f"{cell} ({c.lat:.2f}, {c.lon:.2f})",
                  "gdd_local": gdd_local}
    taw = None if hz is None else sum((h["fc"] - h["wp"]) * h["thickness"] * 10 * (1 - h["gravel"] / 100) for h in hz)
    INFO.append({"cell": cell, "status": "no soil" if hz is None else "run", "sowing_doy": sow, "maturity_doy": mat,
                 "ggcmi_distance_deg": round(dist, 2), "gdd_cultivar": gdd_local,
                 "cultivar_factor": gdd_local / GP["gdd_maturity"],
                 "rock_at_m": (sum(h["thickness"] for h in hz) if hz and hz[-1].get("rock_below") else None) if hz else None,
                 "smu": smu, "soil_depth_m": None if hz is None else sum(h["thickness"] for h in hz),
                 "topsoil": None if hz is None else hz[0]["texture"], "taw_mm": taw,
                 "rain_mm_yr": X["pr"].sum() / (t.size / 365.25), "tmean": float(np.mean((X["tasmax"] + X["tasmin"]) / 2)),
                 "eto_mm_yr": eto.sum() / (t.size / 365.25), "seasons": len(runs),
                 "gdd_season_mean": float(np.mean([r["gdd_season"] for r in runs])),
                 "seasons_below_cultivar_gdd_pct": 100 * np.mean([r["gdd_season"] < gdd_local for r in runs])})
CELLS = CELLS.join(pd.DataFrame(INFO).set_index("cell"))
# a cultivar is a property of the crop: below the earliest cultivar grown today (5th percentile of the local degree-days
# of the cells where maize is grown) the local GGCMI season (taken from the nearest cropped point) is not a cultivar
CULTIVAR_MIN = float(np.percentile(CELLS.loc[CELLS["group"] == "grown", "gdd_cultivar"], 5))
for cell in PREP:
    PREP[cell]["gdd_cultivar"] = int(round(max(PREP[cell]["gdd_local"], CULTIVAR_MIN)))
CELLS["gdd_cultivar_used"] = [PREP[c]["gdd_cultivar"] for c in CELLS.index]
print(f"earliest cultivar: {CULTIVAR_MIN:.0f} degree-days (5th percentile of the grown cells)")
print(f"inputs prepared for {len(PREP)} cells ({time.time() - t0:.0f} s); crop {CROP_FILE}: {GP}")
with pd.option_context("display.width", 250, "display.max_columns", 40):
    display(CELLS.describe().round(2))


def write_inputs(work, cell):
    p = PREP[cell]
    files = aq.write_climate(work / "DATA", cell, p["first"], p["X"]["tasmin"], p["X"]["tasmax"], p["eto"], p["X"]["pr"],
                             title=p["title"])
    aq.write_soil(work / "DATA" / f"{cell}.SOL", p["title"], p["hz"])
    aq.write_crop_variant(CROP_FILE, work / "DATA" / f"{cell}.CRO", p["gdd_cultivar"],
                          title=f"{CROP_FILE[:-4]}, cultivar of {p['gdd_cultivar']} degree-days to maturity ({cell})")
    aq.write_project(work / "LIST" / f"{cell}.PRM", f"{cell} maize {REGION_NAME}", p["runs"], files, f"{cell}.CRO", f"{cell}.SOL")
    return f"{cell}.PRM"

## 3. AquaCrop runs and seasonal results

In [ ]:
t0 = time.time()
torun = CELLS.index[CELLS["status"] == "run"].tolist()
RAW, TIMED_OUT = aq.run_cells(WORK, torun, write_inputs, [CROP_FILE], n_workers=N_WORKERS, timeout=TIMEOUT)
print(f"AquaCrop: {len(torun)} cells in {time.time() - t0:.0f} s ({N_WORKERS} workers); did not end within {TIMEOUT} s: {TIMED_OUT}")
CELLS.loc[TIMED_OUT, "status"] = "no result"
rows = []
for cell in CELLS.index:
    df = RAW.get(cell)
    for k, r in enumerate(PREP[cell]["runs"]):
        row = {"cell": cell, "season": k + 1, "sowing": aq.date_of(r["crop1"]), "gdd_season": r["gdd_season"], "gdd_run": r["gdd_run"]}
        if df is not None and k < len(df):
            row.update(df.iloc[k][["Rain", "ETo", "GD", "Tr", "Cycle", "TempStr", "StoStr", "BioMass", "HI", "Y(dry)", "overflow"]].to_dict())
        rows.append(row)
S = pd.DataFrame(rows)
S["harvest_year"] = [d.year for d in S["sowing"]]
S["yield_t_ha"] = S["Y(dry)"] if "Y(dry)" in S else np.nan                 # AquaCrop's dry yield, as it is
chk = S.dropna(subset=["GD"])                                              # check: our degree-days of the run vs AquaCrop's
print("degree-days of the runs, ours - AquaCrop: median %.1f, max |.| %.1f" % ((chk["gdd_run"] - chk["GD"]).median(),
                                                                                  (chk["gdd_run"] - chk["GD"]).abs().max()))
S = S[S["season"] > 1]                                                     # spin-up season dropped
S.to_csv(OUT / f"seasons_maize_{REGION_NAME}.csv", index=False)
g = S.groupby("cell")
R = pd.DataFrame({"yield_t_ha": g["yield_t_ha"].mean(), "yield_cv": g["yield_t_ha"].std() / g["yield_t_ha"].mean().replace(0, np.nan),
                  "failed_seasons_pct": 100 * g["yield_t_ha"].apply(lambda y: (y < 0.5).mean() if y.notna().any() else np.nan),
                  "cycle_days": g["Cycle"].mean(), "stomata_stress_pct": g["StoStr"].mean(),
                  "temperature_stress_pct": g["TempStr"].mean(), "transpiration_mm": g["Tr"].mean()})
RES = CELLS.join(R)
RES.to_csv(OUT / f"cells_maize_{REGION_NAME}.csv")
grid_vars = ["svm_p", "yield_t_ha", "yield_cv", "failed_seasons_pct", "cycle_days", "stomata_stress_pct",
             "temperature_stress_pct", "transpiration_mm", "gdd_cultivar_used", "taw_mm"]
ds = xr.Dataset({v: (("lat", "lon"), np.full((la.size, lo.size), np.nan, "float32")) for v in grid_vars},
                coords={"lat": la, "lon": lo})
for v in grid_vars:
    ds[v].values[RES["i"].values, RES["j"].values] = pd.to_numeric(RES[v], errors="coerce").values
ds["grown"] = (("lat", "lon"), PRES.astype("int8"))
ds.attrs.update(model=f"AquaCrop {aq.AQ_VERSION}", crop=CROP_FILE, region=f"{REGION_NAME} {REGION}", years=f"{YEARS}")
ds.to_netcdf(OUT / f"aquacrop_maize_{REGION_NAME}.nc")
print(f"status of the cells: {RES['status'].value_counts().to_dict()}")
with pd.option_context("display.width", 250, "display.max_columns", 40):
    display(RES.groupby("group")[["svm_p", "yield_t_ha", "failed_seasons_pct", "cycle_days", "stomata_stress_pct",
                                  "temperature_stress_pct"]].describe().T.round(2))

## 4. Rainfed yield against the SVM probability: maps and comparison

In [ ]:
R2 = RES.dropna(subset=["yield_t_ha"])
rho, prho = stats.spearmanr(R2["svm_p"], R2["yield_t_ha"])
g = R2["group"] == "grown"
auc = lambda x: stats.mannwhitneyu(x[g], x[~g]).statistic / (g.sum() * (~g).sum())
print(f"{len(R2)} cells with a result ({len(RES) - len(R2)} without)")
print(f"Spearman(SVM probability, AquaCrop yield) = {rho:.2f}")
print(f"grown vs not grown today, AUC: AquaCrop yield {auc(R2['yield_t_ha']):.2f}, SVM probability {auc(R2['svm_p']):.2f}")
q = pd.qcut(R2["svm_p"].rank(method="first"), 10, labels=False)
display(R2.groupby(q).agg(svm_p=("svm_p", "median"), yield_t_ha=("yield_t_ha", "median"),
                          failed_pct=("failed_seasons_pct", "median"), cells=("svm_p", "size")).round(2))

ext = [REGION[2], REGION[3], REGION[0], REGION[1]]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, v, cmap, lim, title in ((axes[0], "svm_p", "viridis", (0, 1), f"SVM probability ({cu.REF_LABEL})"),
                                (axes[1], "yield_t_ha", "YlGn", (0, np.nanpercentile(ds["yield_t_ha"], 99)), "AquaCrop rainfed yield (t/ha)")):
    im = ax.pcolormesh(lo, la, ds[v], cmap=cmap, vmin=lim[0], vmax=lim[1], shading="nearest")
    ax.contour(lo, la, ds["grown"].where(np.isfinite(ds[v])).fillna(0), levels=[0.5], colors="r", linewidths=0.6)
    ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.set_title(f"{title}; red: maize grown today", fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)
for grp, col in (("not grown", "0.6"), ("grown", "C2")):
    s_ = R2[R2["group"] == grp]
    axes[2].scatter(s_["svm_p"], s_["yield_t_ha"], s=6, c=col, alpha=0.5, label=f"{grp} today")
axes[2].set_xlabel("SVM probability"); axes[2].set_ylabel("AquaCrop yield (t/ha)"); axes[2].legend(fontsize=8)
axes[2].set_title(f"Spearman {rho:.2f}", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / f"aquacrop_vs_svm_maize_{REGION_NAME}.png", dpi=110); plt.show()